In [ ]:
from dotenv import load_dotenv
import os
from pydantic import BaseModel, Field, EmailStr, field_validator
from langchain_openai import ChatOpenAI , OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.runnables import RunnablePassthrough
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.tools import tool
from langchain.agents import create_agent
import time
from langchain_core.messages import AIMessage




C:\Users\Dell\AppData\Local\Temp\ipykernel_10512\2793816108.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
load_dotenv()

True

Task 1: Structured Profile Generator

In [3]:
class UserProfile (BaseModel):
    name: str = Field(..., description="The user's name")
    age: int = Field(..., description="The user's age")
    email: EmailStr = Field(..., description="The user's Email")
    country: str = Field(..., description="The country where the user lives")
        
    @field_validator("name")
    @classmethod
    def name_validator (cls, v):
        if not v.strip():
            raise ValueError("Name cannot be empty or only whitespace")
        return v.strip()


    @field_validator("country")
    @classmethod
    def country_validator (cls, v):
        if not v.strip():
            raise ValueError("Country cannot be empty or only whitespace")
        return v.strip()
    

In [4]:
prompt = ChatPromptTemplate.from_messages([("system","You are a profile generator. Extract the user's profile as JSON: {format_instructions}"),
                                           ("human","{user_input}")])

llm = ChatOpenAI(model=os.getenv("openai_model"),temperature = 0,max_tokens=500, api_key=os.getenv("openai_key"))

parser = PydanticOutputParser(pydantic_object=UserProfile)

chain = prompt|llm|parser

In [5]:
result1 = chain.invoke({"format_instructions":parser.get_format_instructions(),
                        "user_input":"Ali, I have 27 years old. I'm a software engineer living in Egypt. Ali@gmail.com."})
print(f"Pydantic Model: \n{result1}\n\nStructured JSON: \n{result1.model_dump_json()}")


Pydantic Model: 
name='Ali' age=27 email='Ali@gmail.com' country='Egypt'

Structured JSON: 
{"name":"Ali","age":27,"email":"Ali@gmail.com","country":"Egypt"}


In [30]:
result1.name

'Ali'

In [6]:
result2 = chain.invoke({"format_instructions":parser.get_format_instructions(),
                        "user_input":"I'm a software engineer. I have 30 years old. I am living in Saudi Arabia. I'm Salma . Salma@gmail.com."})
print(f"Pydantic Model: \n{result2}\n\nStructured JSON: \n{result2.model_dump_json()}")

Pydantic Model: 
name='Salma' age=30 email='Salma@gmail.com' country='Saudi Arabia'

Structured JSON: 
{"name":"Salma","age":30,"email":"Salma@gmail.com","country":"Saudi Arabia"}


In [31]:
result2.name

'Salma'

Task 2: Streaming and Batch Concept Explainer

In [7]:
prompt1 = ChatPromptTemplate.from_template("Tell the capital of this country: {country}, using one sentence.")
str_parser = StrOutputParser()
chain = prompt1 |llm | str_parser
for chunk in chain.stream({"country":"Egypt"}):
    print(chunk,end="",flush=True)

The capital of Egypt is Cairo.

In [ ]:
prompt2 = ChatPromptTemplate.from_messages([
    ("system","You will receive a sentence that contains a country name. You should respond with the country's capital. Critical rule: Respond only with one word."),
    ("human","Tell the capital of this country: {country}.")])
    
chain = prompt2 |llm | str_parser

countries = ["Egypt", "Saudi Arabia", "Oman"]
batch_results = chain.batch([{"country":country} for country in countries])


In [9]:
batch_results

['Cairo', 'Riyadh', 'Muscat']

In [10]:
for country, capital in zip(countries, batch_results):
    print (country,":",capital)

Egypt : Cairo
Saudi Arabia : Riyadh
Oman : Muscat


Task 3: Mini RAG Q&A App

In [11]:
loader = TextLoader(file_path="Doc.txt")
Docs = loader.load()
Docs

[Document(metadata={'source': 'Doc.txt'}, page_content='RAG (Retrieval-Augmented Generation) was introduced by Lewis et al. in 2020 to address LLM hallucination and knowledge cutoff.\nVector databases store embeddings for fast Approximate Nearest Neighbor search. HNSW provides O(log N) search with high recall.\nChunking strategy directly impacts retrieval quality. Small chunks (256 tokens) improve precision for QA, while large chunks (1024) preserve context for summarization.\nMetadata filtering allows multi-tenant isolation and ACL. Payload in Qdrant or metadata in Pinecone stores source, page, date.\nHybrid search combines dense (semantic) and sparse (BM25) to improve Recall@10 by 15-25% over dense-only.\nReranking with cross-encoders re-scores top-20 to top-5, adding 80-150ms but improving precision significantly.\nQuery rewriting via Multi-Query and HyDE bridges the query-document gap.\nEvaluation via RAGAS measures Context Precision, Recall, Faithfulness, Answer Relevance.')]

In [12]:
text = Docs[0].page_content
print (text)

RAG (Retrieval-Augmented Generation) was introduced by Lewis et al. in 2020 to address LLM hallucination and knowledge cutoff.
Vector databases store embeddings for fast Approximate Nearest Neighbor search. HNSW provides O(log N) search with high recall.
Chunking strategy directly impacts retrieval quality. Small chunks (256 tokens) improve precision for QA, while large chunks (1024) preserve context for summarization.
Metadata filtering allows multi-tenant isolation and ACL. Payload in Qdrant or metadata in Pinecone stores source, page, date.
Hybrid search combines dense (semantic) and sparse (BM25) to improve Recall@10 by 15-25% over dense-only.
Reranking with cross-encoders re-scores top-20 to top-5, adding 80-150ms but improving precision significantly.
Query rewriting via Multi-Query and HyDE bridges the query-document gap.
Evaluation via RAGAS measures Context Precision, Recall, Faithfulness, Answer Relevance.


In [13]:
splitter = RecursiveCharacterTextSplitter(chunk_size =200, chunk_overlap=40,separators=["\n\n","\n","."," "])
chunks = splitter.split_documents(Docs) 

In [14]:
chunks

[Document(metadata={'source': 'Doc.txt'}, page_content='RAG (Retrieval-Augmented Generation) was introduced by Lewis et al. in 2020 to address LLM hallucination and knowledge cutoff.'),
 Document(metadata={'source': 'Doc.txt'}, page_content='Vector databases store embeddings for fast Approximate Nearest Neighbor search. HNSW provides O(log N) search with high recall.'),
 Document(metadata={'source': 'Doc.txt'}, page_content='Chunking strategy directly impacts retrieval quality. Small chunks (256 tokens) improve precision for QA, while large chunks (1024) preserve context for summarization.'),
 Document(metadata={'source': 'Doc.txt'}, page_content='Metadata filtering allows multi-tenant isolation and ACL. Payload in Qdrant or metadata in Pinecone stores source, page, date.'),
 Document(metadata={'source': 'Doc.txt'}, page_content='Hybrid search combines dense (semantic) and sparse (BM25) to improve Recall@10 by 15-25% over dense-only.'),
 Document(metadata={'source': 'Doc.txt'}, page_co

In [15]:
embedding_model = OpenAIEmbeddings(model =os.getenv("openai_embedding_model"), api_key=os.getenv("openai_key")) 

Faiss_vector_store = FAISS.from_documents(chunks,embedding_model)

In [16]:
Faiss_vector_store.index.ntotal

7

In [17]:
retriever = Faiss_vector_store.as_retriever(search_type="similarity",search_kwargs ={"k":3})

In [18]:
RAG_prompt= ChatPromptTemplate.from_template("""Answer the question based only on the following context {context}
                                            
                                            Question:{question} if you don't know the answer from context ,say the information is not available in the provided document, rather than inventing an answer.""")


RAG_chain =({"context":retriever,"question":RunnablePassthrough()}|RAG_prompt|llm|str_parser)

In [19]:
print(RAG_chain.invoke("What is the rag?"))

RAG (Retrieval-Augmented Generation) was introduced by Lewis et al. in 2020 to address LLM hallucination and knowledge cutoff.


In [20]:
print(RAG_chain.invoke("What problem was RAG introduced to solve?"))

RAG (Retrieval-Augmented Generation) was introduced to address LLM hallucination and knowledge cutoff.


In [21]:
print(RAG_chain.invoke("What is the AI?"))

The information is not available in the provided document.


Task 4: Retriever Comparison Lab

In [22]:
similarity_retriever = Faiss_vector_store.as_retriever(search_type="similarity",search_kwargs ={"k":3})

In [23]:
MMR_retriever = Faiss_vector_store.as_retriever(search_type="mmr",search_kwargs ={"k":3,"fetch_k":10,"lambda_mult":0.7})


In [24]:
bm25 = BM25Retriever.from_documents(chunks)
ensemble_retriever = EnsembleRetriever(retrievers=[bm25,similarity_retriever],weights=[0.3,0.7])

In [25]:
questions = [
    "What is the RAG?",
    "What problem was RAG introduced to solve?",
    "What is the AI?"
]

for question in questions:
    Similarity_Docs_Content = [doc.page_content for doc in similarity_retriever.invoke("What is the RAG?")]
    MMr_Docs_Content = [doc.page_content for doc in MMR_retriever.invoke("What is the RAG?")]
    Ensemble_Docs_Content = [doc.page_content for doc in ensemble_retriever.invoke("What is the RAG?")][:3]
    
    print(f"Question: {question} \n\nSimilarity: \n{"\n".join(Similarity_Docs_Content)} \n\nMMR: \n{"\n".join(MMr_Docs_Content)}\n\nEnsemble: \n{"\n".join(Ensemble_Docs_Content)}\n\n")

Question: What is the RAG? 

Similarity: 
Evaluation via RAGAS measures Context Precision, Recall, Faithfulness, Answer Relevance.
RAG (Retrieval-Augmented Generation) was introduced by Lewis et al. in 2020 to address LLM hallucination and knowledge cutoff.
Metadata filtering allows multi-tenant isolation and ACL. Payload in Qdrant or metadata in Pinecone stores source, page, date. 

MMR: 
Evaluation via RAGAS measures Context Precision, Recall, Faithfulness, Answer Relevance.
RAG (Retrieval-Augmented Generation) was introduced by Lewis et al. in 2020 to address LLM hallucination and knowledge cutoff.
Metadata filtering allows multi-tenant isolation and ACL. Payload in Qdrant or metadata in Pinecone stores source, page, date.

Ensemble: 
Evaluation via RAGAS measures Context Precision, Recall, Faithfulness, Answer Relevance.
Metadata filtering allows multi-tenant isolation and ACL. Payload in Qdrant or metadata in Pinecone stores source, page, date.
RAG (Retrieval-Augmented Generation)

Task5: Tool-UsingAssistant

In [26]:
@tool
def get_current_time():
    """Returns current time. use for questions about current time"""
    return time.time()

@tool
def get_number_of_words(sentence: str):
    """Returns the number of words in a sentence. use when counting words"""
    return len(sentence.split())

@tool
def search_knowledge_base(query: str):
    """Search in vector store Knowledge base. Use for any question about RAG"""
    return Faiss_vector_store.similarity_search(query,k=3)

In [27]:
tools = [get_current_time,get_number_of_words,search_knowledge_base]

agent = create_agent(
    model=llm,
    tools=tools
)

In [28]:
Requests = [
    "What time is it right now?",
    "How many words are in this sentence: 'the quick brown fox jumps over the lazy dog'?",
    "What does the knowledge base say about RAG?"]

In [29]:
for i, request in enumerate(Requests, start=1):
    response = agent.invoke({
    "messages":[{
        "role":"user","content":request
        }]
    })
    
    print (f"- Request {i}: \n  {request}")
    for msg in response["messages"]:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            for tool_call in msg.tool_calls:
                print(f"- Tool Name: \n  {tool_call['name']}")
            
    print (f"- The final answer: \n  {response["messages"][-1].content}\n\n")

- Request 1: 
  What time is it right now?
- Tool Name: 
  get_current_time
- The final answer: 
  The current time is approximately 1792757804 seconds since the epoch. If you need the time in a specific format or timezone, please let me know!


- Request 2: 
  How many words are in this sentence: 'the quick brown fox jumps over the lazy dog'?
- Tool Name: 
  get_number_of_words
- The final answer: 
  The sentence "the quick brown fox jumps over the lazy dog" contains 9 words.


- Request 3: 
  What does the knowledge base say about RAG?
- Tool Name: 
  search_knowledge_base
- The final answer: 
  The knowledge base provides the following information about RAG (Retrieval-Augmented Generation):

1. **Introduction**: RAG was introduced by Lewis et al. in 2020 to address issues related to large language model (LLM) hallucination and knowledge cutoff.

2. **Evaluation**: The evaluation of RAG is conducted via RAGAS, which measures Context Precision, Recall, Faithfulness, and Answer Relevan